# WEEK 3 DAY 4

## Part C - Set Up the Gold Notebook
### Task 4: Create the notebook
#### Settings and helpers:

In [0]:
from pyspark.sql import functions as F

dbutils.widgets.text("storage_account", "stbigdatadev00001")
dbutils.widgets.text("secret_scope", "kv-scope")

sa = dbutils.widgets.get("storage_account")
scope = dbutils.widgets.get("secret_scope")

spark.conf.set("spark.sql.session.timeZone", "UTC")
spark.conf.set(
    f"fs.azure.account.key.{sa}.dfs.core.windows.net",
    dbutils.secrets.get(scope, "storage-account-key"),
)

silver_path = f"abfss://silver@{sa}.dfs.core.windows.net/ecommerce_events"
gold_base = f"abfss://gold@{sa}.dfs.core.windows.net"

def count_if(event):
    return F.sum((F.col("event_type") == event).cast("int"))

def rate(numerator, denominator):
    # percentage; empty when the denominator is 0
    return F.round(F.when(F.col(denominator) > 0, F.col(numerator) / F.col(denominator) * 100), 2)

def write_gold(df, name):
    (df.write.format("delta").mode("overwrite")
       .option("overwriteSchema", "true").save(f"{gold_base}/{name}"))
    print(f"written: {name}")

events = spark.read.format("delta").load(silver_path)

#### Check that silver is readable:

In [0]:
print(events.count())
display(events.limit(5))

109563336


event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session,main_category,event_date
2019-11-01T00:02:32Z,view,58900001,2159649186167915110,unknown,total,15.06,548804259,d8a4a956-1d7a-4ae8-95f1-605402901cef,unknown,2019-11-01
2019-11-01T00:02:35Z,view,16900119,2053013558433677895,furniture.kitchen.table,unknown,57.38,548523725,9f4e068b-d942-40c1-879f-e0de101e6157,furniture,2019-11-01
2019-11-01T00:04:07Z,view,1801766,2053013554415534427,electronics.video.tv,artel,146.85,543653226,ab310b47-1eb2-45f8-8e5b-21ab2010925a,electronics,2019-11-01
2019-11-01T00:05:32Z,view,1003910,2053013555631882655,electronics.smartphone,nokia,118.10,528269305,6dce75d2-e455-4e52-ab65-af16e36f64cb,electronics,2019-11-01
2019-11-01T00:09:16Z,view,1002544,2053013555631882655,electronics.smartphone,apple,458.28,540508491,33a0d8f4-c388-4ce7-85f4-7b1f11563d6a,electronics,2019-11-01


## Part D - Build the Gold Tables
### Task 5: daily_summary
### 

In [0]:
daily = (
    events.groupBy("event_date")
    .agg(
        F.count("*").alias("total_events"),
        count_if("view").alias("views"),
        count_if("cart").alias("carts"),
        count_if("remove_from_cart").alias("cart_removals"),
        count_if("purchase").alias("purchases"),
        F.countDistinct("user_id").alias("unique_users"),
        F.sum(F.when(F.col("event_type") == "purchase", F.col("price")).otherwise(0)).alias("revenue"),
    )
    .withColumn("view_to_purchase_rate", rate("purchases", "views"))
    .withColumn("day_of_week", F.date_format("event_date", "EEEE"))
    .orderBy("event_date")
)
write_gold(daily, "daily_summary")
display(daily)

written: daily_summary


event_date,total_events,views,carts,cart_removals,purchases,unique_users,revenue,view_to_purchase_rate,day_of_week
2019-10-01,1241586,1206088,16193,0,19305,190067,6275579.06,1.6,Tuesday
2019-10-02,1189032,1152692,16871,0,19469,184851,6213628.53,1.69,Wednesday
2019-10-03,1125271,1087314,18702,0,19255,170611,6233782.98,1.77,Thursday
2019-10-04,1414420,1344727,42654,0,27039,209354,8623058.19,2.01,Friday
2019-10-05,1328042,1269990,34560,0,23492,194922,7341094.46,1.85,Saturday
2019-10-06,1316094,1262931,30994,0,22169,193160,6737258.17,1.76,Sunday
2019-10-07,1198138,1159190,17570,0,21378,186867,6348189.06,1.84,Monday
2019-10-08,1364480,1323457,17952,0,23071,221017,6819701.26,1.74,Tuesday
2019-10-09,1341947,1301314,17886,0,22747,213956,6855326.05,1.75,Wednesday
2019-10-10,1280615,1240287,18336,0,21992,205208,6665413.21,1.77,Thursday


### Task 6: category_funnel 

In [0]:
funnel = (
    events.groupBy("main_category")
    .agg(
        count_if("view").alias("views"),
        count_if("cart").alias("carts"),
        count_if("purchase").alias("purchases"),
    )
    .withColumn("view_to_cart_rate", rate("carts", "views"))
    .withColumn("cart_to_purchase_rate", rate("purchases", "carts"))
    .withColumn("view_to_purchase_rate", rate("purchases", "views"))
    .orderBy(F.desc("views"))
)
write_gold(funnel, "category_funnel")
display(funnel)

written: category_funnel


main_category,views,carts,purchases,view_to_cart_rate,cart_to_purchase_rate,view_to_purchase_rate
electronics,36995544,2126480,916616,5.75,43.1,2.48
unknown,33941940,900962,407625,2.65,45.24,1.2
appliances,12826086,428642,174011,3.34,40.6,1.36
computers,6279627,141547,62330,2.25,44.03,0.99
apparel,4449553,52357,22217,1.18,42.43,0.5
furniture,3277316,42258,19841,1.29,46.95,0.61
auto,2150852,46740,21338,2.17,45.65,0.99
construction,1756458,45132,16500,2.57,36.56,0.94
kids,1289524,22544,11648,1.75,51.67,0.9
accessories,614939,7878,3741,1.28,47.49,0.61


### Task 7: Reference table

In [0]:
display(events.select("main_category").distinct().orderBy("main_category"))

main_category
accessories
apparel
appliances
auto
computers
construction
country_yard
electronics
furniture
kids


#### Unmapped categories to Other

In [0]:
mapping = [
    ("electronics", "Tech"), ("computers", "Tech"),
    ("appliances", "Home"), ("furniture", "Home"),
    ("country_yard", "Home"), ("construction", "Home"),
    ("apparel", "Fashion"), ("accessories", "Fashion"),
    ("kids", "Family"), ("medicine", "Family"),
    ("sport", "Lifestyle"), ("auto", "Lifestyle"), ("stationery", "Lifestyle"),
]
ref = spark.createDataFrame(mapping, ["main_category", "category_group"])
write_gold(ref, "ref_category_group")

written: ref_category_group


### Task 8: revenue_by_category_brand (with the join)

In [0]:
revenue = (
    events.filter(F.col("event_type") == "purchase")
    .withColumn("event_month", F.date_format("event_date", "yyyy-MM"))
    .groupBy("event_month", "main_category", "brand")
    .agg(
        F.count("*").alias("purchases"),
        F.sum("price").alias("revenue"),
        F.round(F.avg("price"), 2).alias("avg_price"),
    )
    .join(F.broadcast(ref), "main_category", "left")
    .fillna({"category_group": "Other"})
)
write_gold(revenue, "revenue_by_category_brand")
display(revenue.orderBy(F.desc("revenue")).limit(20))

written: revenue_by_category_brand


main_category,event_month,brand,purchases,revenue,avg_price,category_group
electronics,2019-11,apple,164188,125113740.92,762.02,Tech
electronics,2019-10,apple,141379,109372259.42,773.61,Tech
electronics,2019-11,samsung,182631,50677070.76,277.48,Tech
electronics,2019-10,samsung,159603,43188709.17,270.60,Tech
electronics,2019-11,xiaomi,55673,10227139.45,183.70,Tech
electronics,2019-10,xiaomi,44760,8333024.77,186.17,Tech
unknown,2019-11,unknown,49807,7271696.29,146.00,Other
unknown,2019-10,unknown,38294,5375229.31,140.37,Other
electronics,2019-10,huawei,23218,4869419.79,209.73,Tech
electronics,2019-11,huawei,23395,4766830.85,203.75,Tech


### Task 9: hourly_traffic

In [0]:
hourly = (
    events.withColumn("event_hour", F.hour("event_time"))
    .groupBy("event_hour")
    .agg(
        F.count("*").alias("total_events"),
        count_if("view").alias("views"),
        count_if("purchase").alias("purchases"),
        F.countDistinct("event_date").alias("active_days"),
    )
    .withColumn("avg_events_per_day", F.round(F.col("total_events") / F.col("active_days"), 0))
    .orderBy("event_hour")
)
write_gold(hourly, "hourly_traffic")
display(hourly)

written: hourly_traffic


event_hour,total_events,views,purchases,active_days,avg_events_per_day
0,753437,727592,5771,61,12351.0
1,1397688,1346691,9741,61,22913.0
2,2736110,2625286,25124,61,44854.0
3,3935408,3731313,55968,61,64515.0
4,4966081,4681982,85232,61,81411.0
5,5550072,5221605,101429,61,90985.0
6,5801551,5454298,109427,61,95107.0
7,5898220,5541735,112109,61,96692.0
8,6056811,5687386,120454,61,99292.0
9,5983999,5610955,126616,61,98098.0


### Task 10: product_performance

In [0]:
def known(col):
    # most common pattern: pick a real value over 'unknown'
    return F.coalesce(F.max(F.when(F.col(col) != "unknown", F.col(col))), F.lit("unknown"))

products = (
    events.groupBy("product_id")
    .agg(
        known("brand").alias("brand"),
        known("main_category").alias("main_category"),
        count_if("view").alias("views"),
        count_if("cart").alias("carts"),
        count_if("purchase").alias("purchases"),
        F.sum(F.when(F.col("event_type") == "purchase", F.col("price")).otherwise(0)).alias("revenue"),
    )
    .filter(F.col("views") >= 100)
    .withColumn("view_to_purchase_rate", rate("purchases", "views"))
)
write_gold(products, "product_performance")
display(products.orderBy(F.desc("views")).limit(20))

written: product_performance


product_id,brand,main_category,views,carts,purchases,revenue,view_to_purchase_rate
1004856,samsung,electronics,942081,126852,61259,7917145.03,6.5
1005115,apple,electronics,910697,78629,34785,33030410.27,3.82
1004767,samsung,electronics,861628,98134,44417,11004247.78,5.16
4804056,apple,electronics,497413,75640,30179,4921326.24,6.07
1005105,apple,electronics,473629,31891,15776,21684603.37,3.33
1004870,samsung,electronics,462510,52166,21288,6057424.03,4.6
1004249,apple,electronics,462344,38587,17970,13544675.45,3.89
1004833,samsung,electronics,450439,53475,26183,4494994.25,5.81
1005160,xiaomi,electronics,437323,28771,9310,1930487.09,2.13
1002544,apple,electronics,409147,42553,22225,10457979.14,5.43


## Part E - Optimize and Validate
### Task 11: Look at file counts before OPTIMIZE

In [0]:
tables = ["daily_summary", "category_funnel", "revenue_by_category_brand",
          "hourly_traffic", "product_performance", "ref_category_group"]

def num_files(name):
    return spark.sql(f"DESCRIBE DETAIL delta.`{gold_base}/{name}`").select("numFiles").first()[0]

before = {t: num_files(t) for t in tables}
print(before)

{'daily_summary': 1, 'category_funnel': 1, 'revenue_by_category_brand': 1, 'hourly_traffic': 1, 'product_performance': 5, 'ref_category_group': 1}


### Task 12: Run OPTIMIZE

In [0]:
for t in tables:
    if t == "product_performance":
        spark.sql(f"OPTIMIZE delta.`{gold_base}/{t}` ZORDER BY (product_id)")
    else:
        spark.sql(f"OPTIMIZE delta.`{gold_base}/{t}`")

after = {t: num_files(t) for t in tables}
print(after)

{'daily_summary': 1, 'category_funnel': 1, 'revenue_by_category_brand': 1, 'hourly_traffic': 1, 'product_performance': 1, 'ref_category_group': 1}


### Task 13: Cross-check the numbers

In [0]:
silver_rows = events.count()
silver_revenue = events.filter("event_type = 'purchase'").agg(F.sum("price")).first()[0]

d = spark.read.format("delta").load(f"{gold_base}/daily_summary")
r = spark.read.format("delta").load(f"{gold_base}/revenue_by_category_brand")

print("silver rows:               ", silver_rows)
print("daily_summary events:      ", d.agg(F.sum("total_events")).first()[0])
print("silver revenue:            ", silver_revenue)
print("daily_summary revenue:     ", d.agg(F.sum("revenue")).first()[0])
print("revenue_by_cat_brand:      ", r.agg(F.sum("revenue")).first()[0])

silver rows:                109563336
daily_summary events:       109563336
silver revenue:             505126293.44
daily_summary revenue:      505126293.44
revenue_by_cat_brand:       505126293.44


### Task 14: Answer for business questions

####  temporary views

In [0]:
for t in tables:
    spark.read.format("delta").load(f"{gold_base}/{t}").createOrReplaceTempView(t)

#### Which categories earn the most revenue?

In [0]:
%sql
SELECT category_group, main_category, SUM(revenue) AS revenue
FROM revenue_by_category_brand GROUP BY 1, 2 ORDER BY revenue DESC LIMIT 10

category_group,main_category,revenue
Tech,electronics,381695319.29
Other,unknown,52801355.37
Home,appliances,32222145.73
Tech,computers,25372201.51
Home,furniture,4216494.03
Lifestyle,auto,2648960.53
Home,construction,2013385.72
Fashion,apparel,1805962.87
Family,kids,1401903.93
Lifestyle,sport,718251.75


#### Which brands earn the most revenue (excluding unknown)?

In [0]:
%sql
SELECT brand, SUM(revenue) AS revenue
FROM revenue_by_category_brand WHERE brand <> 'unknown' GROUP BY 1 ORDER BY revenue DESC LIMIT 10

brand,revenue
apple,238712020.58
samsung,101271688.32
xiaomi,20452486.15
huawei,9663787.08
lg,8626379.67
acer,6923022.22
lucente,6649742.79
sony,6341082.98
oppo,5900574.74
lenovo,4450744.83


#### Which products are viewed a lot but rarely bought?

In [0]:
%sql
SELECT product_id, brand, main_category, views, purchases, view_to_purchase_rate
FROM product_performance ORDER BY views DESC LIMIT 20

product_id,brand,main_category,views,purchases,view_to_purchase_rate
1004856,samsung,electronics,942081,61259,6.5
1005115,apple,electronics,910697,34785,3.82
1004767,samsung,electronics,861628,44417,5.16
4804056,apple,electronics,497413,30179,6.07
1005105,apple,electronics,473629,15776,3.33
1004870,samsung,electronics,462510,21288,4.6
1004249,apple,electronics,462344,17970,3.89
1004833,samsung,electronics,450439,26183,5.81
1005160,xiaomi,electronics,437323,9310,2.13
1002544,apple,electronics,409147,22225,5.43


#### Busiest hours (UTC)

In [0]:
%sql
SELECT event_hour, avg_events_per_day FROM hourly_traffic ORDER BY avg_events_per_day DESC LIMIT 5

event_hour,avg_events_per_day
16,123620.0
15,121422.0
17,116822.0
14,114578.0
13,102317.0


## END 
#### saved as `gold_ecommerce` and exported as `gold_ecommerce.ipynb`